# Trax : Ungraded Lecture Notebook

In this notebook you'll get to know about the Trax framework and learn about some of its basic building blocks.

## Background

### Why Trax and not TensorFlow or PyTorch?

TensorFlow and PyTorch are both extensive frameworks that can do almost anything in deep learning. They offer a lot of flexibility, but that often means verbosity of syntax and extra time to code.

Trax is much more concise. It runs on a TensorFlow backend but allows you to train models with 1 line commands. Trax also runs end to end, allowing you to get data, model and train all with a single terse statements. This means you can focus on learning, instead of spending hours on the idiosyncrasies of big framework implementation.

### Why not Keras then?

Keras is now part of Tensorflow itself from 2.0 onwards. Also, trax is good for implementing new state of the art algorithms like Transformers, Reformers, BERT because it is actively maintained by Google Brain Team for advanced deep learning tasks. It runs smoothly on CPUs,GPUs and TPUs as well with comparatively lesser modifications in code.

### How to Code in Trax
Building models in Trax relies on 2 key concepts:- **layers** and **combinators**.
Trax layers are simple objects that process data and perform computations. They can be chained together into composite layers using Trax combinators, allowing you to build layers and models of any complexity.

### Trax, JAX, TensorFlow and Tensor2Tensor

You already know that Trax uses Tensorflow as a backend, but it also uses the JAX library to speed up computation too. You can view JAX as an enhanced and optimized version of numpy. 

**Watch out for assignments which import `import trax.fastmath.numpy as np`. If you see this line, remember that when calling `np` you are really calling Trax’s version of numpy that is compatible with JAX.**

As a result of this, where you used to encounter the type `numpy.ndarray` now you will find the type `jax.interpreters.xla.DeviceArray`.

Tensor2Tensor is another name you might have heard. It started as an end to end solution much like how Trax is designed, but it grew unwieldy and complicated. So you can view Trax as the new improved version that operates much faster and simpler.

### Resources

- Trax source code can be found on Github: [Trax](https://github.com/google/trax)
- JAX library: [JAX](https://jax.readthedocs.io/en/latest/index.html)


## Installing Trax

Trax has dependencies on JAX and some libraries like JAX which are yet to be supported in [Windows](https://github.com/google/jax/blob/1bc5896ee4eab5d7bb4ec6f161d8b2abb30557be/README.md#installation) but work well in Ubuntu and MacOS. We would suggest that if you are working on Windows, try to install Trax on WSL2. 

Official maintained documentation - [trax-ml](https://trax-ml.readthedocs.io/en/latest/) not to be confused with this [TraX](https://trax.readthedocs.io/en/latest/index.html)

In [1]:
import numpy as np  # regular ol' numpy

from trax import layers as tl  # core building block
from trax import shapes  # data signatures: dimensionality and type
from trax import fastmath  # uses jax, offers numpy on steroids

In [2]:
# Trax version 1.3.9 or better 
# !pip list | grep trax

'grep' is not recognized as an internal or external command,
operable program or batch file.


## Layers
Layers are the core building blocks in Trax or as mentioned in the lectures, they are the base classes.

They take inputs, compute functions/custom calculations and return outputs.

You can also inspect layer properties. Let me show you some examples.

### Relu Layer
First let's see how to build a relu activation function as a layer. A layer like this is one of the simplest types. Notice there is no object initialization so it works just like a math function.

**Note: Activation functions are also layers in Trax, which might look odd if you have been using other frameworks for a longer time.**

In [3]:
# layers
# create a relu trax layer
relu = tl.Relu()

# Inspect properties
print("-- Properties --")
print("name :", relu.name)
print("expected inputs :", relu.n_in)
print("promised outputs :", relu.n_out, "\n")

# Inputs
x = np.array([-2, -1, 0, 1, 2])
print("-- Inputs --")
print("x :", x, "\n")

# Outputs
y = relu(x)
print("-- Outputs --")
print("y :", y)

No GPU/TPU found, falling back to CPU. (Set TF_CPP_MIN_LOG_LEVEL=0 and rerun for more info.)


-- Properties --
name : Serial
expected inputs : 1
promised outputs : 1 

-- Inputs --
x : [-2 -1  0  1  2] 

-- Outputs --
y : [0 0 0 1 2]


In [4]:
# Create a concatenate trax layer
concat = tl.Concatenate()
print("-- Properties --")
print("name :", concat.name)
print("expected inputs :", concat.n_in)
print("promised outputs :", concat.n_out, "\n")

# Inputs
x1 = np.array([-10, -20, -30])
x2 = x1 / -10
print("-- Inputs --")
print("x1 :", x1)
print("x2 :", x2, "\n")

# Outputs
y = concat([x1, x2])
print("-- Outputs --")
print("y :", y)

-- Properties --
name : Concatenate
expected inputs : 2
promised outputs : 1 

-- Inputs --
x1 : [-10 -20 -30]
x2 : [1. 2. 3.] 

-- Outputs --
y : [-10. -20. -30.   1.   2.   3.]


In [5]:
# Configure a concatenate layer
concat_3 = tl.Concatenate(n_items=3)  # configure the layer's expected inputs
print("-- Properties --")
print("name :", concat_3.name)
print("expected inputs :", concat_3.n_in)
print("promised outputs :", concat_3.n_out, "\n")

# Inputs
x1 = np.array([-10, -20, -30])
x2 = x1 / -10
x3 = x2 * 0.99
print("-- Inputs --")
print("x1 :", x1)
print("x2 :", x2)
print("x3 :", x3, "\n")

# Outputs
y = concat_3([x1, x2, x3])
print("-- Outputs --")
print("y :", y)

-- Properties --
name : Concatenate
expected inputs : 3
promised outputs : 1 

-- Inputs --
x1 : [-10 -20 -30]
x2 : [1. 2. 3.]
x3 : [0.99 1.98 2.97] 

-- Outputs --
y : [-10.   -20.   -30.     1.     2.     3.     0.99   1.98   2.97]


## Layers can have Weights
Some layer types include mutable weights and biases that are used in computation and training. Layers of this type require initialization before use.

For example the `LayerNorm` layer calculates normalized data, that is also scaled by weights and biases. During initialization you pass the data shape and data type of the inputs, so the layer can initialize compatible arrays of weights and biases.

In [6]:
# Layer initialization
norm = tl.LayerNorm()
# You first must know what the input data will look like
x = np.array([0, 1, 2, 3], dtype="float")

# Use the input data signature to get shape and type for initializing weights and biases
norm.init(shapes.signature(x)) # We need to convert the input datatype from usual tuple to trax ShapeDtype

print("Normal shape:",x.shape, "Data Type:",type(x.shape))
print("Shapes Trax:",shapes.signature(x),"Data Type:",type(shapes.signature(x)))

# Inspect properties
print("-- Properties --")
print("name :", norm.name)
print("expected inputs :", norm.n_in)
print("promised outputs :", norm.n_out)
# Weights and biases
print("weights :", norm.weights[0])
print("biases :", norm.weights[1], "\n")

# Inputs
print("-- Inputs --")
print("x :", x)

# Outputs
y = norm(x)
print("-- Outputs --")
print("y :", y)

Normal shape: (4,) Data Type: <class 'tuple'>
Shapes Trax: ShapeDtype{shape:(4,), dtype:float64} Data Type: <class 'trax.shapes.ShapeDtype'>
-- Properties --
name : LayerNorm
expected inputs : 1
promised outputs : 1
weights : [1. 1. 1. 1.]
biases : [0. 0. 0. 0.] 

-- Inputs --
x : [0. 1. 2. 3.]
-- Outputs --
y : [-1.3416404  -0.44721344  0.44721344  1.3416404 ]


c:\Users\mrinm\anaconda3\envs\NLP_env\lib\site-packages\trax\layers\normalization.py:141: UserWarning: Explicitly requested dtype float64 requested in ones is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/google/jax#current-gotchas for more.
  scale = jnp.ones(features, dtype=input_signature.dtype)
c:\Users\mrinm\anaconda3\envs\NLP_env\lib\site-packages\trax\layers\normalization.py:142: UserWarning: Explicitly requested dtype float64 requested in zeros is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/google/jax#current-gotchas for more.
  bias = jnp.zeros(features, dtype=input_signature.dtype)


## Custom Layers
This is where things start getting more interesting!
You can create your own custom layers too and define custom functions for computations by using `tl.Fn`. Let me show you how.

In [7]:
# Define a custom layer
# In this example you will create a layer to calculate the input times 2

def TimesTwo():
    layer_name = "TimesTwo" #don't forget to give your custom layer a name to identify

    # Custom function for the custom layer
    def func(x):
        return x * 2

    return tl.Fn(layer_name, func)


# Test it
times_two = TimesTwo()

# Inspect properties
print("-- Properties --")
print("name :", times_two.name)
print("expected inputs :", times_two.n_in)
print("promised outputs :", times_two.n_out, "\n")

# Inputs
x = np.array([1, 2, 3])
print("-- Inputs --")
print("x :", x, "\n")

# Outputs
y = times_two(x)
print("-- Outputs --")
print("y :", y)

-- Properties --
name : TimesTwo
expected inputs : 1
promised outputs : 1 

-- Inputs --
x : [1 2 3] 

-- Outputs --
y : [2 4 6]


## Combinators
You can combine layers to build more complex layers. Trax provides a set of objects named combinator layers to make this happen. Combinators are themselves layers, so behavior commutes.

### Serial Combinator
This is the most common and easiest to use. For example could build a simple neural network by combining layers into a single layer using the `Serial` combinator. This new layer then acts just like a single layer, so you can inspect intputs, outputs and weights. Or even combine it into another layer! Combinators can then be used as trainable models. _Try adding more layers_

**Note: As you must have guessed, if there is serial combinator, there must be a parallel combinator as well. Do try to explore about combinators and other layers from the trax documentation and look at the repo to understand how these layers are written.**


In [8]:
# Serial combinator
serial = tl.Serial(
    tl.LayerNorm(),         # normalize input
    tl.Relu(),              # convert negative values to zero
    times_two,              # the custom layer you created above, multiplies the input recieved from above by 2
    
    ## START CODE HERE
    tl.Dense(n_units=2),  # try adding more layers. eg uncomment these lines
    tl.Dense(n_units=1),  # Binary classification, maybe? uncomment at your own peril
    tl.LogSoftmax()       # Yes, LogSoftmax is also a layer
    ## END CODE HERE
)

# Initialization
x = np.array([-2, -1, 0, 1, 2]) #input
serial.init(shapes.signature(x)) #initialising serial instance

print("-- Serial Model --")
print(serial,"\n")
print("-- Properties --")
print("name :", serial.name)
print("sublayers :", serial.sublayers)
print("expected inputs :", serial.n_in)
print("promised outputs :", serial.n_out)
print("weights & biases:", serial.weights, "\n")

# Inputs
print("-- Inputs --")
print("x :", x, "\n")

# Outputs
y = serial(x)
print("-- Outputs --")
print("y :", y)

-- Serial Model --
Serial[
  LayerNorm
  Serial[
    Relu
  ]
  TimesTwo
  Dense_2
  Dense_1
  LogSoftmax
] 

-- Properties --
name : Serial
sublayers : [LayerNorm, Serial[
  Relu
], TimesTwo, Dense_2, Dense_1, LogSoftmax]
expected inputs : 1
promised outputs : 1
weights & biases: ((Array([1, 1, 1, 1, 1], dtype=int32), Array([0, 0, 0, 0, 0], dtype=int32)), ((), (), ()), (), (Array([[ 0.0049592 ,  0.38593373],
       [ 0.7077786 ,  0.34572968],
       [ 0.48597485,  0.5944897 ],
       [ 0.84076184,  0.27473697],
       [-0.58620316, -0.29388332]], dtype=float32), Array([ 8.0278244e-07, -8.7016070e-07], dtype=float32)), (Array([[-0.49807122],
       [-0.8271764 ]], dtype=float32), Array([-1.4532857e-06], dtype=float32)), ()) 

-- Inputs --
x : [-2 -1  0  1  2] 

-- Outputs --
y : [0.]


## JAX
Just remember to lookout for which numpy you are using, the regular ol' numpy or Trax's JAX compatible numpy. Both tend to use the alias np so watch those import blocks.

**Note: There are certain things which are still not possible in fastmath.numpy which can be done in numpy so you will see in assignments we will switch between them to get our work done.**

In [9]:
# Numpy vs fastmath.numpy have different data types
# Regular ol' numpy
x_numpy = np.array([1, 2, 3])
print("good old numpy : ", type(x_numpy), "\n")

# Fastmath and jax numpy
x_jax = fastmath.numpy.array([1, 2, 3])
print("jax trax numpy : ", type(x_jax))

good old numpy :  <class 'numpy.ndarray'> 

jax trax numpy :  <class 'jaxlib.xla_extension.ArrayImpl'>


# Classes and subclasses 

In this notebook, I will show you the basics of classes and subclasses in Python. As you've seen in the lectures from this week, `Trax` uses layer classes as building blocks for deep learning models, so it is important to understand how classes and subclasses behave in order to be able to build custom layers when needed. 

By completing this notebook, you will:

- Be able to define classes and subclasses in Python
- Understand how inheritance works in subclasses
- Be able to work with instances

# Part 1: Parameters, methods and instances

First, let's define a class `My_Class`. 

In [10]:
class My_Class: 
    def __init__(self, y, z): #Initialization of x_1 and x_2 with arguments y and z
        self.x_1 = y
        self.x_2 = z
        
    def __call__(self):       #When called, adds the values of parameters x_1 and x_2, prints and returns the result 
        result = self.x_1 + self.x_2 
        print("Addition of {} and {} is {}".format(self.x_1,self.x_2,result))
        
        return result

In [11]:
instance_e = My_Class(10,15)
def test_class_definition():
    
    assert instance_e.x_1 == 10, "Check the value assigned to x_1"
    assert instance_e.x_2 == 15, "Check the value assigned to x_2"
    assert instance_e() == 25, "Check the __call__ method"
    
    print("\033[92mAll tests passed!")
    
test_class_definition()

Addition of 10 and 15 is 25
All tests passed!


In [12]:
class My_Class: 
    def __init__(self, y, z): #Initialization of x_1 and x_2 with arguments y and z
        self.x_1 = y
        self.x_2 = z
    def __call__(self):       #Performs an operation with x_1 and x_2, and returns the result
        a = self.x_1 - 2*self.x_2 
        return a
    def my_method(self, w):   #Multiplies x_1 and x_2, adds argument w and returns the result
        result = self.x_1*self.x_2 + w
        return result

In [13]:
instance_f = My_Class(1,10)
print("Output of my_method:",instance_f.my_method(16))

Output of my_method: 26


In [14]:
# hidden-cell
class My_Class: 
    def __init__(self, y, z):      #Initialization of x_1 and x_2 with arguments y and z
        self.x_1 = y
        self.x_2 = z
    def __call__(self):            #Performs an operation with x_1 and x_2, and returns the result
        a = self.x_1 - 2*self.x_2 
        return a
    def my_method(self, w):        #Multiplies x_1 and x_2, adds argument w and returns the result
        b = self.x_1*self.x_2 + w
        return b
    def new_method(self, v):       #Calls My_method with argument v
        result = self.my_method(v)
        return result

In [15]:
instance_g = My_Class(1,10)
print("Output of my_method:",instance_g.my_method(16))
print("Output of new_method:",instance_g.new_method(16))

Output of my_method: 26
Output of new_method: 26


# Part 2: Subclasses and Inheritance

`Trax` uses classes and subclasses to define layers. The base class in `Trax` is `layer`, which means that every layer from a deep learning model is defined as a subclass of the `layer` class. In this part of the notebook, you are going to see how subclasses work. To define a subclass `sub` from class `super`, you have to write `class sub(super):` and define any method and parameter that you want for your subclass. In the next cell, I define `sub_c` as a subclass of `My_Class` with only one method (`additional_method`).

In [16]:
class sub_c(My_Class):           #Subclass sub_c from My_class
    def additional_method(self): #Prints the value of parameter x_1
        print(self.x_1)

## 2.1 Inheritance

When you define a subclass `sub`, every method and parameter is inherited from `super` class, including the `__init__` and `__call__` methods. This means that any instance from `sub` can use the methods defined in `super`.  Run the following cell and see for yourself.

In [17]:
instance_sub_a = sub_c(1,10)
print('Parameter x_1 of instance_sub_a: ' + str(instance_sub_a.x_1))
print('Parameter x_2 of instance_sub_a: ' + str(instance_sub_a.x_2))
print("Output of my_method of instance_sub_a:",instance_sub_a.my_method(16))


Parameter x_1 of instance_sub_a: 1
Parameter x_2 of instance_sub_a: 10
Output of my_method of instance_sub_a: 26


# Data generators

In Python, a generator is a function that behaves like an iterator. It will return the next item. Here is a [link](https://wiki.python.org/moin/Generators) to review python generators. In many AI applications, it is advantageous to have a data generator to handle loading and transforming data for different applications. 

You will now implement a custom data generator, using a common pattern that you will use during all assignments of this course.
In the following example, we use a set of samples `a`, to derive a new set of samples, with more elements than the original set.

**Note: Pay attention to the use of list `lines_index` and variable `index` to traverse the original list.**

In [18]:
import random as rnd
import numpy as np

In [19]:
a = [1, 2, 3, 4]
b = [0] * 10
a_size = len(a)
b_size = len(b)
lines_index = [*range(a_size)] 
print(lines_index)

[0, 1, 2, 3]


In [20]:
index = 0
for i in range(b_size):
    if index >= a_size:
        index = 0
    b[i] = a[lines_index[index]]
    index += 1

print(b)

[1, 2, 3, 4, 1, 2, 3, 4, 1, 2]


## Shuffling the data order

In the next example, we will do the same as before, but shuffling the order of the elements in the output list. Note that here, our strategy of traversing using `lines_index` and `index` becomes very important, because we can simulate a shuffle in the input data, without doing that in reality.


In [21]:
# Example of traversing a list of indexes to create a circular list
a = [1, 2, 3, 4]
b = []
a_size = len(a)
b_size = 10

lines_index = [*range(a_size)]  # list of indexes fpr a 

# if we shuffle the index_list we can change the order of our circular list
# without modifying the order or our original data

rnd.shuffle(lines_index)
print("Shuffled order of index:",lines_index)

print("New value order for first batch:", [a[index] for index in lines_index])

batch_counter = 1
index = 0
for i in range(b_size):
    # wrap by resetting index to zero if b is longer than a
    if index >= a_size:
        index = 0
        batch_counter += 1
        rnd.shuffle(lines_index) # Re-shuffle the order
        print("\nShuffled Indexes for Batch No.{} :{}".format(batch_counter,lines_index))
        print("Values for Batch No.{} :{}".format(batch_counter,[a[index] for index in lines_index]))
    
    b.append(a[lines_index[index]])     #  `indexes_list[index]` point to a index of a. Store the result in b
    index += 1
print()
print('Final value of b:',b)


Shuffled order of index: [1, 2, 0, 3]
New value order for first batch: [2, 3, 1, 4]

Shuffled Indexes for Batch No.2 :[3, 0, 1, 2]
Values for Batch No.2 :[4, 1, 2, 3]

Shuffled Indexes for Batch No.3 :[0, 1, 2, 3]
Values for Batch No.3 :[1, 2, 3, 4]

Final value of b: [2, 3, 1, 4, 4, 1, 2, 3, 1, 2]


### Exercise

**Instructions:** Implement a data generator function that takes in `batch_size, x, y shuffle` where x could be a large list of samples, and y is a list of the tags associated with those samples. Return a subset of those inputs in a tuple of two arrays `(X,Y)`. Each is an array of dimension (`batch_size`). If `shuffle=True`, the data will be traversed in a random form.

In [22]:
def data_generator(batch_size, data_x, data_y, shuffle = True):
    '''
      Input: 
        batch_size - integer describing the batch size
        data_x - list containing samples
        data_y - list containing labels
        shuffle - Shuffle the data order
      Output:
        a tuple containing 2 elements:
        X - list of dim (batch_size) of samples
        Y - list of dim (batch_size) of labels
    '''
    data_lng = len(data_x) # len(data_x) must be equal to len(data_y)
    index_list = [*range(data_lng)] # Create a list with the ordered indexes of sample data

    # If shuffle is set to true, we traverse the list in a random way
    if shuffle:
        rnd.shuffle(index_list) # Inplace shuffle of the list
    
    index = 0 # Start with the first element

    while True:
        X = [0]*batch_size # We can create a list with batch_size elements.
        Y = [0]*batch_size # We can create a list with batch_size elements.

        for i in range(batch_size): 
            # Wrap the index each time that we reach the end of the list
            if index >= data_lng:
                index = 0
                if shuffle:
                    rnd.shuffle(index_list) # Shuffle the index_list if shuffle is true

            X[i] = data_x[index]
            Y[i] = data_y[index]

            index += 1
        
        yield((X, Y))

In [23]:
def test_data_generator():
    x = [1, 2, 3, 4]
    y = [xi ** 2 for xi in x]
    
    generator = data_generator(3, x, y, shuffle=False)

    assert np.allclose(next(generator), ([1, 2, 3], [1, 4, 9])),  "First batch does not match"
    assert np.allclose(next(generator), ([4, 1, 2], [16, 1, 4])), "Second batch does not match"
    assert np.allclose(next(generator), ([3, 4, 1], [9, 16, 1])), "Third batch does not match"
    assert np.allclose(next(generator), ([2, 3, 4], [4, 9, 16])), "Fourth batch does not match"

    print("\033[92mAll tests passed!")

test_data_generator()

All tests passed!


In [24]:
# Example
class Myclass(object):

    def __init__(self, y):
        self.y = y
    
    def my_method(self, x):
        return x + self.y
    
    def __call__(self, x):
        return self.my_method(x)
    

f = Myclass(12)
print(f(2))
    

14
